# Imports

In [1]:
import asyncio
import logging
from typing import Optional, List, Dict, Any, Union
from abc import ABC, abstractmethod
from enum import Enum
from dataclasses import dataclass, field
from datetime import datetime
from pydantic import BaseModel, Field, field_validator, ConfigDict
from langchain_ollama import OllamaLLM
from codeshield.cs import CodeShield

# Models

In [2]:
class SeverityLevel(str, Enum):
    """
    Enumeration of security issue severity levels.
    
    Attributes:
        CRITICAL: Critical security vulnerability requiring immediate action
        HIGH: High severity issue that should be addressed urgently
        MEDIUM: Medium severity issue that should be reviewed
        LOW: Low severity issue for awareness
        INFO: Informational finding without direct security impact
    """
    CRITICAL = "critical"
    HIGH = "high"
    MEDIUM = "medium"
    LOW = "low"
    INFO = "info"

In [3]:
class TreatmentRecommendation(str, Enum):
    """
    Enumeration of recommended treatments for security issues.
    
    Attributes:
        BLOCK: Block the code execution completely
        WARN: Allow with warning message
        ALLOW: Allow without restrictions
    """
    BLOCK = "block"
    WARN = "warn"
    ALLOW = "allow"

In [4]:
class SecurityIssue(BaseModel):
    """
    Represents a single security issue found in code.
    
    Attributes:
        pattern_id: Unique identifier for the security pattern detected
        description: Human-readable description of the security issue
        severity: Severity level of the issue
        line: Line number where the issue was found
        column: Optional column number for more precise location
        cwe_id: Optional Common Weakness Enumeration identifier
    """
    model_config = ConfigDict(use_enum_values=True)
    
    pattern_id: str = Field(..., min_length=1, description="Unique pattern identifier")
    description: str = Field(..., min_length=1, description="Issue description")
    severity: SeverityLevel = Field(..., description="Severity level")
    line: int = Field(..., ge=1, description="Line number (1-indexed)")
    column: Optional[int] = Field(None, ge=1, description="Column number (1-indexed)")
    cwe_id: Optional[str] = Field(None, description="CWE identifier")
    
    @field_validator('pattern_id', 'description')
    @classmethod
    def validate_non_empty_strings(cls, v: str) -> str:
        """Validate that string fields are not empty or whitespace only."""
        if not v or not v.strip():
            raise ValueError("Field cannot be empty or whitespace only")
            
        return v.strip()

In [5]:
class ScanResult(BaseModel):
    """
    Represents the complete result of a code security scan.
    
    Attributes:
        is_insecure: Whether security issues were detected
        recommended_treatment: Recommended action to take
        issues_found: List of security issues detected
        scan_timestamp: When the scan was performed
        code_hash: Optional hash of scanned code for tracking
    """
    model_config = ConfigDict(use_enum_values=True)
    
    is_insecure: bool = Field(..., description="Whether code is insecure")
    recommended_treatment: TreatmentRecommendation = Field(..., description="Treatment recommendation")
    issues_found: List[SecurityIssue] = Field(default_factory=list, description="List of issues")
    scan_timestamp: datetime = Field(default_factory=datetime.now, description="Scan timestamp")
    code_hash: Optional[str] = Field(None, description="Hash of scanned code")
    
    @field_validator('issues_found')
    @classmethod
    def validate_issues_consistency(cls, v: List[SecurityIssue], info) -> List[SecurityIssue]:
        """Validate that issues list is consistent with is_insecure flag."""
        is_insecure = info.data.get('is_insecure', False)
        if is_insecure and len(v) == 0:
            print("Code marked as insecure but no issues found")
            
        return v

In [6]:
class LLMResponse(BaseModel):
    """
    Represents the response from LLM with security validation.
    
    Attributes:
        original_code: Original code generated by LLM
        treated_code: Code after applying security treatments
        scan_result: Results of security scan
        is_safe: Whether the code passed security checks
    """
    original_code: str = Field(..., min_length=1, description="Original LLM output")
    treated_code: str = Field(..., description="Code after security treatment")
    scan_result: ScanResult = Field(..., description="Security scan results")
    is_safe: bool = Field(..., description="Whether code is safe to use")
    
    @field_validator('original_code', 'treated_code')
    @classmethod
    def validate_code_not_empty(cls, v: str) -> str:
        """Validate that code fields are not empty."""
        if not v or not v.strip():
            raise ValueError("Code cannot be empty")
        
        return v

# Security Scanner

In [7]:
class BaseSecurityScanner(ABC):
    """
    Abstract base class for security scanners.
    
    Defines the interface that all security scanner implementations must follow.
    """
    
    @abstractmethod
    async def scan_code(self, code: str) -> ScanResult:
        """
        Scan code for security vulnerabilities.
        
        Args:
            code: Source code to scan
            
        Returns:
            ScanResult containing scan findings
            
        Raises:
            ValueError: If code is invalid
            RuntimeError: If scan fails
        """
        pass
    
    @abstractmethod
    def sanitize_input(self, code: str) -> str:
        """
        Sanitize code input before scanning.
        
        Args:
            code: Raw code input
            
        Returns:
            Sanitized code string
        """
        pass

In [9]:
class CodeShieldSecurityScanner(BaseSecurityScanner):
    """
    Security scanner implementation using CodeShield.

    Attributes:
        max_code_length: Maximum allowed code length (default: 100,000 chars)
        timeout: Scan timeout in seconds (default: 30)
    """

    def __init__(self, max_code_length: int = 100_000, timeout: int = 30):
        """
        Initialize the security scanner.

        Args:
            max_code_length: Maximum allowed code length
            timeout: Timeout for scan operations in seconds

        Raises:
            ValueError: If parameters are invalid
        """
        if max_code_length <= 0:
            raise ValueError("max_code_length must be positive")

        if timeout <= 0:
            raise ValueError("timeout must be positive")

        self.max_code_length = max_code_length
        self.timeout = timeout
        print(f"Initialized CodeShieldSecurityScacnner (max_length={max_code_length}, timeout={timeout})")

    def sanitize_input(self, code: str) -> str:
        """
        Sanitize  and validate code input.

        Args:
            code: Raw code input

        Returns:
            Sanitized code string

        Raises:
            ValueError: If code is invalid or too long
        """
        if not code or not code.strip():
            raise ValueError("Code cannot be empty or whitespace only")

        code = code.strip()

        if len(code) > self.max_code_length:
            raise ValueError(
                f"Code length ({len(code)} exceeds maximum allowed length ({self.max_code_length})"
            )

        # Remove potentially dangerous control characters but keep newlines and tabs
        sanitized = ''.join(char for char in code if char.isprintable() or char in '\n\r\t')
        print(f"Sanitized code: {len(sanitized)} characters")
        return sanitized

    async def scan_code(self, code: str) -> ScanResult:
        """
        Scan code for security vulnerabilities using CodeShield.

        Args:
            code: Source code to scan

        Returns:
            ScanResult with security findings

        Raises:
            ValueError: If code is invalid
            RuntimeError: If scan operation fails
            TimeoutError: If scan exceeds timeout
        """
        try:
            # Sanitize input
            sanitized_code = self.sanitize_input(code)
            print("Starting security scan...")

            # Perform scan with timeout
            try:
                result = await asyncio.wait_for(
                    CodeShield.scan_code(sanitized_code),
                    timeout=self.timeout
                )

            except asyncio.TimeoutError:
                print(f"Security scan timed out after {self.timeout} seconds")
                raise TimeoutError(f"Security scan timed out after {self.timeout} seconds")

            # Transform result to our data model
            scan_result = self._transform_result(result)

            print(
                f"Scan complete: {'INSECURE' if scan_result.is_insecure else 'SECURE'} "
                f"({len(scan_result.issues_found)} issues found)"
            )

            return scan_result

        except ValueError as e:
            print(f"Validation error during scan: {e}")
            raise

        except TimeoutError as e:
            raise

        except Exception as e:
            print(f"Unexpected error during security scan: {e}", exc_info=True)
            raise RuntimeError(f"Security scan failed: {str(e)}") from e

    def _transform_result(self, raw_result: Any) -> ScanResult:
        """
        Transform CodeShield result to ScanResult model.
        
        Args:
            raw_result: Raw result from CodeShield
            
        Returns:
            Validated ScanResult instance
            
        Raises:
            ValueError: If result transformation fails
        """
        try:
            # Extract issues from raw result
            issues: List[SecurityIssue] = []
            
            if hasattr(raw_result, 'issues_found') and raw_result.issues_found:
                for raw_issue in raw_result.issues_found:
                    try:
                        # Get severity from raw issue and map it to SeverityLevel
                        raw_severity = getattr(raw_issue, 'severity', None)
                        severity = SeverityLevel.MEDIUM  # Default

                        if raw_severity:
                            # Convert to string and map to SeverityLevel
                            severity_str = str(raw_severity).lower()
                            if 'critical' in severity_str or 'error' in severity_str:
                                severity = SeverityLevel.CRITICAL
                            elif 'high' in severity_str:
                                severity = SeverityLevel.HIGH
                            elif 'medium' in severity_str or 'warning' in severity_str:
                                severity = SeverityLevel.MEDIUM
                            elif 'low' in severity_str or 'info' in severity_str:
                                severity = SeverityLevel.LOW

                        issue = SecurityIssue(
                            pattern_id=getattr(raw_issue, 'pattern_id', 'unknown'),
                            description=getattr(raw_issue, 'description', 'No description'),
                            severity=severity,
                            line=getattr(raw_issue, 'line', 1),
                            column=getattr(raw_issue, 'column', None),
                            cwe_id=getattr(raw_issue, 'cwe_id', None)
                        )
                        issues.append(issue)
                    except Exception as e:
                        print(f"Failed to parse issue: {e}")
                        continue
            
            # Determine treatment recommendation
            treatment = TreatmentRecommendation.ALLOW
            if hasattr(raw_result, 'recommended_treatment'):
                treatment_str = raw_result.recommended_treatment.lower()
                if treatment_str in [t.value for t in TreatmentRecommendation]:
                    treatment = TreatmentRecommendation(treatment_str)
            
            # Create ScanResult
            scan_result = ScanResult(
                is_insecure=getattr(raw_result, 'is_insecure', False),
                recommended_treatment=treatment,
                issues_found=issues,
                scan_timestamp=datetime.now()
            )
            
            return scan_result
            
        except Exception as e:
            print(f"Failed to transform scan result: {e}")
            raise ValueError(f"Invalid scan result format: {str(e)}") from e

# Initialize LLM

In [10]:
class LLMConfig(BaseModel):
    """
    Configuration for LLM initialization.
    
    Attributes:
        model_name: Name of the Ollama model to use
        base_url: Base URL for Ollama API
        temperature: Sampling temperature (0.0 to 1.0)
        timeout: Request timeout in seconds
        max_tokens: Maximum tokens in response
    """
    model_name: str = Field(default="llama2", min_length=1, description="Ollama model name")
    base_url: str = Field(default="http://localhost:11434", description="Ollama API URL")
    temperature: float = Field(default=0.7, ge=0.0, le=1.0, description="Sampling temperature")
    timeout: int = Field(default=60, ge=1, description="Request timeout in seconds")
    max_tokens: int = Field(default=1000, ge=1, description="Maximum tokens in response")
    
    @field_validator('base_url')
    @classmethod
    def validate_url(cls, v: str) -> str:
        """Validate that base_url is properly formatted."""
        if not v.startswith(('http://', 'https://')):
            raise ValueError("base_url must start with http:// or https://")
        
        return v.rstrip('/')

In [11]:
def initialize_llm(config: Optional[LLMConfig] = None) -> OllamaLLM:
    """
    Initialize LangChain Ollama LLM with configuration validation.
    
    Args:
        config: Optional LLM configuration. Uses defaults if not provided.
        
    Returns:
        Configured OllamaLLM instance
        
    Raises:
        ValueError: If configuration is invalid
        RuntimeError: If LLM initialization fails
    """
    try:
        # Use default config if not provided
        if config is None:
            config = LLMConfig()
        
        print(f"Initializing Ollama LLM with model: {config.model_name}")
        
        # Initialize LLM with secure settings
        llm = OllamaLLM(
            model=config.model_name,
            base_url=config.base_url,
            temperature=config.temperature,
            timeout=config.timeout,
            num_predict=config.max_tokens
        )
        
        print("LLM initialized successfully.")
        return llm
        
    except Exception as e:
        print(f"Failed to initialize LLM: {e}", exc_info=True)
        raise RuntimeError(f"LLM initialization failed: {str(e)}") from e

In [12]:
def test_llm_connection(llm: OllamaLLM, timeout: int = 10) -> bool:
    """
    Test LLM connection with a simple query.
    
    Args:
        llm: Initialized LLM instance
        timeout: Test timeout in seconds
        
    Returns:
        True if connection successful, False otherwise
    """
    try:
        print("Testing LLM connection...")
        test_prompt = "Say 'OK' if you can hear me."
        
        # Test with timeout
        response = llm.invoke(test_prompt, timeout=timeout)
        
        if response:
            print("LLM connection test successful")
            return True
        else:
            print("LLM returned empty response")
            return False
            
    except Exception as e:
        print(f"LLM connection test failed: {e}")
        return False

In [14]:
llm_config = LLMConfig(
    model_name="llama3.2:3b",
    temperature=0.3,
    max_tokens=2000
)

In [15]:
llm = initialize_llm(llm_config)
print(f"LLM initialized: {llm_config.model_name}")
print(f"Base URL: {llm_config.base_url}")
print(f"Temperature: {llm_config.temperature}")

Initializing Ollama LLM with model: llama3.2:3b
LLM initialized successfully.
LLM initialized: llama3.2:3b
Base URL: http://localhost:11434
Temperature: 0.3


# Generate Code with LLM

In [16]:
@dataclass
class CodeGenerationRequest:
    """
    Request for code generation from LLM.
    
    Attributes:
        prompt: User prompt for code generation
        max_length: Maximum allowed prompt length
        language: Programming language for generated code
    """
    prompt: str
    max_length: int = 5000
    language: str = "python"
    
    def __post_init__(self):
        """Validate the request after initialization."""
        if not self.prompt or not self.prompt.strip():
            raise ValueError("Prompt cannot be empty")
        
        if len(self.prompt) > self.max_length:
            raise ValueError(f"Prompt exceeds maximum length of {self.max_length}")
        
        self.prompt = self.prompt.strip()

In [17]:
def sanitize_prompt(prompt: str) -> str:
    """
    Sanitize user prompt to prevent injection attacks.
    
    Args:
        prompt: Raw user prompt
        
    Returns:
        Sanitized prompt
    """
    # Remove potentially dangerous characters
    dangerous_chars = ['`', '$', '\\', '<', '>']
    sanitized = prompt
    
    for char in dangerous_chars:
        sanitized = sanitized.replace(char, '')
    
    # Limit consecutive newlines
    while '\n\n\n' in sanitized:
        sanitized = sanitized.replace('\n\n\n', '\n\n')
    
    return sanitized.strip()

In [18]:
async def generate_code_with_llm(
    llm: OllamaLLM,
    request: CodeGenerationRequest,
    timeout: int = 60
) -> str:
    """
    Generate code using LLM with comprehensive error handling.
    
    Args:
        llm: Initialized LLM instance
        request: Code generation request
        timeout: Generation timeout in seconds
        
    Returns:
        Generated code string
        
    Raises:
        ValueError: If request is invalid
        TimeoutError: If generation exceeds timeout
        RuntimeError: If generation fails
    """
    try:
        # Sanitize prompt
        sanitized_prompt = sanitize_prompt(request.prompt)
        
        # Construct secure prompt for code generation
        full_prompt = f"""Generate {request.language} code for the following task. 
Only provide the code without explanations or markdown formatting.

Task: {sanitized_prompt}

Code:"""
        
        print(f"Generating code with prompt length: {len(full_prompt)}")
        
        # Generate code with timeout
        try:
            # Run LLM invocation in executor to support async timeout
            loop = asyncio.get_event_loop()
            code = await asyncio.wait_for(
                loop.run_in_executor(None, llm.invoke, full_prompt),
                timeout=timeout
            )
        except asyncio.TimeoutError:
            print(f"Code generation timed out after {timeout} seconds")
            raise TimeoutError(f"Code generation timed out after {timeout} seconds")
        
        if not code or not code.strip():
            raise RuntimeError("LLM returned empty code")
        
        # Clean up the response
        code = code.strip()
        
        # Remove markdown code blocks if present
        if code.startswith('```'):
            lines = code.split('\n')
            code = '\n'.join(lines[1:-1] if len(lines) > 2 else lines[1:])
            code = code.strip()
        
        print(f"✓ Code generated successfully ({len(code)} characters)")
        return code
        
    except ValueError as e:
        print(f"Invalid request: {e}")
        raise
    except TimeoutError as e:
        raise
    except Exception as e:
        print(f"Code generation failed: {e}", exc_info=True)
        raise RuntimeError(f"Failed to generate code: {str(e)}") from e

In [39]:
def _log_scan_results(response: LLMResponse) -> None:
    """
    Log comprehensive scan results.
    
    Args:
        response: LLM response with scan results
    """
    scan_result = response.scan_result
    
    print("\n" + "=" * 80)
    print("## LLM OUTPUT SECURITY ANALYSIS")
    print("=" * 80)
    
    # Overall status
    status = "INSECURE" if scan_result.is_insecure else "SECURE"
    print(f"\nStatus: {status}")
    print(f"Safe to use: {'Yes' if response.is_safe else 'No'}")
    print(f"Recommended treatment: {scan_result.recommended_treatment.upper()}")
    print(f"Scan timestamp: {scan_result.scan_timestamp.isoformat()}")
    
    # Issues summary
    print(f"\n## Security Issues Found: {len(scan_result.issues_found)}")
    
    if scan_result.issues_found:
        print("\nDetailed Findings:")
        print("-" * 80)
        
        for idx, issue in enumerate(scan_result.issues_found, 1):
            print(f"\n{idx}. {issue.severity.upper()} - {issue.pattern_id}")
            print(f"   Description: {issue.description}")
            print(f"   Location: Line {issue.line}" + (f", Column {issue.column}" if issue.column else ""))
            if issue.cwe_id:
                print(f"   CWE ID: {issue.cwe_id}")
    else:
        print("No security issues detected.")
    
    # Code preview
    print("\n" + "=" * 80)
    print("## Treated Code Preview")
    print("=" * 80)
    preview_lines = response.treated_code.split('\n')[:10]
    print('\n'.join(preview_lines))
    
    print("\n" + "=" * 80 + "\n")

# Scan LLM Output for Security Issues

In [40]:
async def scan_llm_output(
    scanner: BaseSecurityScanner,
    llm_output_code: str
) -> LLMResponse:
    """
    Scan LLM-generated code for security vulnerabilities and apply treatments.
    
    This function performs comprehensive security scanning on LLM-generated code,
    applies appropriate security treatments based on findings, and returns a
    complete response with security analysis.
    
    Args:
        scanner: Security scanner instance to use
        llm_output_code: Code generated by LLM to scan
        
    Returns:
        LLMResponse containing original code, treated code, and scan results
        
    Raises:
        ValueError: If input code is invalid
        RuntimeError: If scanning fails
    """
    try:
        print("=" * 80)
        print("Starting LLM output security scan")
        print("=" * 80)
        
        # Validate input
        if not llm_output_code or not llm_output_code.strip():
            raise ValueError("Cannot scan empty code")
        
        original_code = llm_output_code.strip()
        
        # Perform security scan
        scan_result = await scanner.scan_code(original_code)
        
        # Apply security treatment based on recommendation
        treated_code = apply_security_treatment(original_code, scan_result)
        
        # Determine if code is safe to use
        is_safe = not scan_result.is_insecure or scan_result.recommended_treatment != TreatmentRecommendation.BLOCK
        
        # Create response
        response = LLMResponse(
            original_code=original_code,
            treated_code=treated_code,
            scan_result=scan_result,
            is_safe=is_safe
        )
        
        # Log results
        _log_scan_results(response)
        
        return response
        
    except ValueError as e:
        print(f"Validation error: {e}")
        raise
    except Exception as e:
        print(f"Scan failed: {e}", exc_info=True)
        raise RuntimeError(f"Security scan failed: {str(e)}") from e

In [41]:
def apply_security_treatment(code: str, scan_result: ScanResult) -> str:
    """
    Apply security treatment to code based on scan results.
    
    Args:
        code: Original code
        scan_result: Security scan results
        
    Returns:
        Treated code string
    """
    if not scan_result.is_insecure:
        # Code is secure, no treatment needed
        return code
    
    treatment = scan_result.recommended_treatment
    
    if treatment == TreatmentRecommendation.BLOCK:
        # Block the code completely
        return (
            "# *** SECURITY VIOLATION: Code blocked due to critical security issues ***\n"
            "# The generated code contains severe security vulnerabilities and cannot be executed.\n"
            "# Please review the security findings and regenerate the code.\n"
            f"# Issues found: {len(scan_result.issues_found)}\n"
            "# Original code has been blocked for security reasons.\n"
        )
    
    elif treatment == TreatmentRecommendation.WARN:
        # Add warning but keep code
        warning_header = (
            "# *** SECURITY WARNING ***\n"
            "# This code contains security vulnerabilities. Use with extreme caution.\n"
            f"# {len(scan_result.issues_found)} security issue(s) detected.\n"
            "# Review the findings below before using this code.\n"
            "# " + "-" * 70 + "\n\n"
        )
        
        warning_footer = (
            "\n\n# " + "-" * 70 + "\n"
            "# *** END OF CODE WITH SECURITY WARNINGS ***\n"
        )
        
        return warning_header + code + warning_footer
    
    else:
        # Allow without modification
        return code

In [42]:
scanner = CodeShieldSecurityScanner(max_code_length=100_000, timeout=30)

Initialized CodeShieldSecurityScacnner (max_length=100000, timeout=30)


# 1 - Scanning insecure code

In [47]:
insecure_code = """
def get_md5_sum(s):
    return hashlib.md5(s)
"""

In [48]:
# Run the security scan
result1 = await scan_llm_output(scanner, insecure_code)

Starting LLM output security scan
Sanitized code: 45 characters
Starting security scan...
Scan complete: INSECURE (3 issues found)

## LLM OUTPUT SECURITY ANALYSIS

Status: INSECURE
Safe to use: Yes
Recommended treatment: WARN
Scan timestamp: 2025-12-08T20:50:45.050362

## Security Issues Found: 3

Detailed Findings:
--------------------------------------------------------------------------------

1. MEDIUM - weak-md5-hashing
   Description: Use of weak hashing algorithm
   Location: Line 2
   CWE ID: CWE-327

2. MEDIUM - risky-crypto-algorithm
   Description: Use of a Broken or Risky Cryptographic Algorithm
   Location: Line 2
   CWE ID: CWE-327

3. MEDIUM - insecure-md5-hash-usage
   Description: The MD5 hash function is considered insecure. Avoid using it unless explicitly needed for compatibility reasons
   Location: Line 2
   CWE ID: CWE-328

## Treated Code Preview
# *** SECURITY WARNING ***
# This code contains security vulnerabilities. Use with extreme caution.
# 3 security iss

# 2- Scanning secure code

In [49]:
secure_code = """
def get_sha256_sum(s: str) -> str:
    return hashlib.sha256(s.encode()).hexdigest()
"""

In [50]:
result2 = await scan_llm_output(scanner, secure_code)

Starting LLM output security scan
Sanitized code: 84 characters
Starting security scan...
Scan complete: SECURE (0 issues found)

## LLM OUTPUT SECURITY ANALYSIS

Status: SECURE
Safe to use: Yes
Recommended treatment: ALLOW
Scan timestamp: 2025-12-08T20:50:49.749507

## Security Issues Found: 0
No security issues detected.

## Treated Code Preview
def get_sha256_sum(s: str) -> str:
    return hashlib.sha256(s.encode()).hexdigest()




# 3 - Full LLM Integration Pipeline

In [51]:
async def secure_code_generation_pipeline(
    llm: OllamaLLM,
    scanner: BaseSecurityScanner,
    prompt: str,
    max_retries: int = 3
) -> Optional[LLMResponse]:
    """
    Complete pipeline for secure code generation with LLM.
    
    This function implements a secure workflow that:
    1. Validates and sanitizes user input
    2. Generates code using LLM
    3. Scans generated code for security issues
    4. Applies appropriate security treatments
    5. Retries if blocked (optional)
    
    Args:
        llm: Initialized LLM instance
        scanner: Security scanner instance
        prompt: User prompt for code generation
        max_retries: Maximum retry attempts for blocked code
        
    Returns:
        LLMResponse if successful, None if all retries fail
        
    Raises:
        ValueError: If inputs are invalid
        RuntimeError: If pipeline fails critically
    """
    try:
        print("\n" + "=" * 80)
        print("SECURE CODE GENERATION PIPELINE")
        print("=" * 80)
        
        # Create and validate request
        request = CodeGenerationRequest(prompt=prompt, language="python")
        
        retry_count = 0
        while retry_count < max_retries:
            try:
                # Step 1: Generate code with LLM
                print(f"\n[Attempt {retry_count + 1}/{max_retries}] Generating code...")
                generated_code = await generate_code_with_llm(llm, request, timeout=60)
                
                # Step 2: Scan for security issues
                print("Scanning generated code for security issues...")
                response = await scan_llm_output(scanner, generated_code)
                
                # Step 3: Check if code is safe
                if response.is_safe:
                    print("✓ Code generation successful and secure!")
                    return response
                
                # Step 4: Handle blocked code
                if response.scan_result.recommended_treatment == TreatmentRecommendation.BLOCK:
                    print(f"Code blocked due to security issues. Retry {retry_count + 1}/{max_retries}")
                    retry_count += 1
                    
                    if retry_count < max_retries:
                        # Modify prompt to request more secure code
                        request.prompt += " Make sure to use secure coding practices and avoid deprecated cryptographic methods."
                        continue
                else:
                    # Code has warnings but is allowed
                    print("Code has security warnings but is allowed")
                    return response
                    
            except TimeoutError as e:
                print(f"Timeout occurred: {e}")
                retry_count += 1
                if retry_count >= max_retries:
                    raise
                continue
        
        # All retries exhausted
        print("All retry attempts exhausted. Could not generate secure code.")
        return None
        
    except ValueError as e:
        print(f"Invalid input: {e}")
        raise
    except Exception as e:
        print(f"Pipeline failed: {e}", exc_info=True)
        raise RuntimeError(f"Secure code generation pipeline failed: {str(e)}") from e

In [52]:
# User prompt requesting code generation
user_prompt = "Write a Python function to hash a password string"

In [53]:
# Run secure code generation pipeline
final_result = await secure_code_generation_pipeline(
    llm=llm,
    scanner=scanner,
    prompt=user_prompt,
    max_retries=3
)


SECURE CODE GENERATION PIPELINE

[Attempt 1/3] Generating code...
Generating code with prompt length: 176
Code generated successfully (779 characters)
Scanning generated code for security issues...
Starting LLM output security scan
Sanitized code: 779 characters
Starting security scan...
Scan complete: SECURE (0 issues found)

## LLM OUTPUT SECURITY ANALYSIS

Status: SECURE
Safe to use: Yes
Recommended treatment: ALLOW
Scan timestamp: 2025-12-08T20:51:51.125572

## Security Issues Found: 0
No security issues detected.

## Treated Code Preview
import hashlib
import secrets

def hash_password(password):
    salt = secrets.token_hex(16)
    hashed_password = hashlib.pbkdf2_hmac('sha256', password.encode('utf-8'), 
                                       salt.encode('utf-8'), 100000)
    return salt + ':' + hashed_password.hex()

def verify_password(stored_hash, provided_password):


✓ Code generation successful and secure!


In [54]:
if final_result:
    print("\n### Final Generated Code:")
    print(final_result.treated_code)
    print(f"\n### Security Status: {'SAFE' if final_result.is_safe else 'UNSAFE'}")
else:
    print("\nFailed to generate secure code after all retry attempts")


### Final Generated Code:
import hashlib
import secrets

def hash_password(password):
    salt = secrets.token_hex(16)
    hashed_password = hashlib.pbkdf2_hmac('sha256', password.encode('utf-8'), 
                                       salt.encode('utf-8'), 100000)
    return salt + ':' + hashed_password.hex()

def verify_password(stored_hash, provided_password):
    salt = stored_hash.split(':')[0]
    stored_hash = stored_hash.split(':')[1]
    new_hash = hashlib.pbkdf2_hmac('sha256', provided_password.encode('utf-8'), 
                                   salt.encode('utf-8'), 100000)
    return new_hash == int(stored_hash, 16)

# Example usage:
password = "mysecretpassword"
hashed_password = hash_password(password)
print(hashed_password)

is_valid = verify_password(hashed_password, password)

### Security Status: SAFE
